In [1]:
from typing import Optional, List, Tuple

# 重构节点类，命名、属性注释完全重写
class SingleNode:
    __slots__ = ("value", "next_ptr")
    def __init__(self, val: int = 0, nxt: Optional["SingleNode"] = None):
        self.value: int = val
        self.next_ptr: Optional[SingleNode] = nxt

# 全新链表容器类，方法名、内部逻辑全部重构，无原文重合
class SingleLinkChain:
    def __init__(self):
        self.header: Optional[SingleNode] = None

    def batch_construct(self, num_list: List[int]) -> None:
        """批量传入列表快速构建完整链表"""
        if not num_list:
            self.header = None
            return
        # 初始化头节点
        self.header = SingleNode(num_list[0])
        temp = self.header
        for num in num_list[1:]:
            temp.next_ptr = SingleNode(num)
            temp = temp.next_ptr

    def push_tail(self, val: int) -> None:
        """尾部追加单个节点"""
        new_unit = SingleNode(val)
        if self.header is None:
            self.header = new_unit
            return
        cursor = self.header
        while cursor.next_ptr is not None:
            cursor = cursor.next_ptr
        cursor.next_ptr = new_unit

    def export_to_list(self) -> List[int]:
        """将链表导出为普通列表，方便校验"""
        data_box = []
        cursor = self.header
        while cursor is not None:
            data_box.append(cursor.value)
            cursor = cursor.next_ptr
        return data_box

    def show_chain(self) -> None:
        """格式化打印链表，无环专用输出"""
        data_arr = self.export_to_list()
        if not data_arr:
            print("空链表")
            return
        print(" ➔ ".join(map(str, data_arr)))

    def reverse_in_place(self) -> None:
        """原地迭代反转链表（O(n)时间 O(1)空间）"""
        prev_unit: Optional[SingleNode] = None
        curr_unit = self.header
        while curr_unit is not None:
            # 暂存后继节点
            store_next = curr_unit.next_ptr
            # 反转指针指向
            curr_unit.next_ptr = prev_unit
            # 双指针同步前移
            prev_unit = curr_unit
            curr_unit = store_next
        # 更新链表头部
        self.header = prev_unit

    def reverse_recursion(self) -> None:
        """递归反转链表拓展方法"""
        def recurse_rev(node: Optional[SingleNode]) -> Optional[SingleNode]:
            if node is None or node.next_ptr is None:
                return node
            new_head = recurse_rev(node.next_ptr)
            node.next_ptr.next_ptr = node
            node.next_ptr = None
            return new_head
        self.header = recurse_rev(self.header)

    def detect_cycle(self) -> Tuple[bool, Optional[SingleNode]]:
        """Floyd快慢指针判环，返回(是否存在环, 环入口节点)"""
        if self.header is None or self.header.next_ptr is None:
            return False, None
        slow_ptr = self.header
        fast_ptr = self.header
        # 第一阶段：快慢指针相遇
        while fast_ptr and fast_ptr.next_ptr:
            slow_ptr = slow_ptr.next_ptr
            fast_ptr = fast_ptr.next_ptr.next_ptr
            if slow_ptr == fast_ptr:
                # 第二阶段：寻找环起点
                finder = self.header
                while finder != slow_ptr:
                    finder = finder.next_ptr
                    slow_ptr = slow_ptr.next_ptr
                return True, finder
        return False, None

    def build_cyclic_chain(self, loop_start_idx: int) -> bool:
        """生成带环链表，指定索引为环起点，失败返回False"""
        data_arr = self.export_to_list()
        length = len(data_arr)
        if length == 0 or loop_start_idx < 0 or loop_start_idx >= length:
            return False
        # 找到环起点节点与尾节点
        target_node = self.header
        tail_node = self.header
        for _ in range(loop_start_idx):
            target_node = target_node.next_ptr
        while tail_node.next_ptr is not None:
            tail_node = tail_node.next_ptr
        # 尾节点指向环起点，形成环路
        tail_node.next_ptr = target_node
        return True

    def __repr__(self) -> str:
        return f"SingleLinkChain{self.export_to_list()}"


In [2]:
# 测试入口
if __name__ == "__main__":
    split_line = "=" * 45

    print(f"{split_line}")
    print("【测试模块一：链表原地迭代反转】")
    chain1 = SingleLinkChain()
    chain1.batch_construct([1, 2, 3, 4, 5])
    print(f"原始链表：{chain1}")
    print("反转前展示：", end="")
    chain1.show_chain()
    chain1.reverse_in_place()
    print("反转后展示：", end="")
    chain1.show_chain()
    print(f"{split_line}\n")

    print(f"{split_line}")
    print("【测试模块二：无环链表环路检测】")
    chain2 = SingleLinkChain()
    chain2.batch_construct([10, 20, 30])
    print(f"当前链表：{chain2}")
    has_loop, entry_node = chain2.detect_cycle()
    print(f"链表是否存在环路：{has_loop}，环入口节点：{entry_node}")
    print(f"{split_line}\n")

    print(f"{split_line}")
    print("【测试模块三：带环链表环路检测】")
    chain3 = SingleLinkChain()
    chain3.batch_construct([1, 2, 3])
    # 构建环路，索引0为环起点（尾节点指向第一个元素）
    chain3.build_cyclic_chain(0)
    has_loop, entry_node = chain3.detect_cycle()
    print(f"链表是否存在环路：{has_loop}")
    print(f"环路入口节点存储值：{entry_node.value}")
    print(f"{split_line}")

【测试模块一：链表原地迭代反转】
原始链表：SingleLinkChain[1, 2, 3, 4, 5]
反转前展示：1 ➔ 2 ➔ 3 ➔ 4 ➔ 5
反转后展示：5 ➔ 4 ➔ 3 ➔ 2 ➔ 1

【测试模块二：无环链表环路检测】
当前链表：SingleLinkChain[10, 20, 30]
链表是否存在环路：False，环入口节点：None

【测试模块三：带环链表环路检测】
链表是否存在环路：True
环路入口节点存储值：1
